In [0]:
dbutils.widgets.text("mysql_host","")
dbutils.widgets.text("mysql_database","")
dbutils.widgets.text("mysql_user","")
dbutils.widgets.text("mysql_password","")
dbutils.widgets.text("traget_table","")

dbutils.widgets.text("source_tables","orders,transactions,products")
dbutils.widgets.text("watermark_column","order_date")

In [0]:
mysql_host = dbutils.widgets.get("mysql_host")
mysql_database = dbutils.widgets.get("mysql_database")
mysql_user = dbutils.widgets.get("mysql_user")
mysql_password = dbutils.widgets.get("mysql_password")
traget_table = dbutils.widgets.get("traget_table")
source_tables = dbutils.widgets.get("source_tables").split(",")
watermark_column = dbutils.widgets.get("watermark_column")

In [0]:
from pyspark.sql.functions import current_timestamp, to_date, col
from delta.tables import DeltaTable
import logging

logging.basicConfig(level=logging.INFO)
def log(msg):
    print(f"[INFO] {msg}")


# JDBC connection details
jdbc_url = f"jdbc:sqlserver://{mysql_host}.database.windows.net:1433;database={mysql_database}"
connection_properties = {
    "user": mysql_user,
    "password": mysql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Name of the column that tracks last updates
watermark_column = "order_date"


# Function to get last watermark from RAW table
def get_last_watermark(table):
    raw_table = f"ddddddddddd.raw.{table}"
    if spark.catalog.tableExists(raw_table):
        last_value = spark.sql(f"""
            SELECT MAX({watermark_column}) AS last_value
            FROM {raw_table}
        """).collect()[0]["last_value"]
        log(f"Last watermark for {table}: {last_value}")
        return last_value
    return None


# Read incremental records from source
def read_incremental(table):
    last_value = get_last_watermark(table)
    if last_value:
        query = f"(SELECT * FROM {table} WHERE {watermark_column} > TRY_CAST('{last_value}' AS datetime2)) as src"
        log(f"Reading incremental data for {table}")
    else:
        query = f"(SELECT * FROM {table}) as src"
        log(f"First load for {table}")

    df = spark.read.jdbc(
        url=jdbc_url,
        table=query,
        properties=connection_properties
    )
    return df


# Load RAW table (append-only)
def load_raw(table, df):
    raw_table = f"ddddddddddd.raw.{table}"
    raw_df = df.withColumn("ingestion_time", current_timestamp())
    raw_df.write \
        .format("delta") \
        .mode("append") \
        .option("path", f"abfss://divyacontainer@divyastorrage.dfs.core.windows.net/raw_{table}") \
        .saveAsTable(raw_table)
    log(f"Loaded RAW table {raw_table}")


# Upsert into Bronze (incremental + updated)
def upsert_bronze(table, df):
    bronze_table_name = f"ddddddddddd.bronze.{table}"
    bronze_path = f"abfss://divyacontainer@divyastorrage.dfs.core.windows.net/bronze_{table}"

    df = df.withColumn("partition_date", to_date(current_timestamp()))

    if spark.catalog.tableExists(bronze_table_name):
        target = DeltaTable.forName(spark, bronze_table_name)
        # Upsert into Bronze for your source columns
        target.alias("t").merge(
            df.alias("s"),
            "t.order_id = s.order_id"
        ).whenMatchedUpdate(
            condition="t.product_id <> s.product_id OR t.customer_id <> s.customer_id OR \
                    t.quantity <> s.quantity OR t.price <> s.price OR t.order_date <> s.order_date",
            set={
                "product_id": "s.product_id",
                "customer_id": "s.customer_id",
                "quantity": "s.quantity",
                "price": "s.price",
                "order_date": "s.order_date",
                "partition_date": "s.partition_date"
            }
        ).whenNotMatchedInsertAll().execute()
        log(f"Upserted Bronze table {bronze_table_name}")
    else:
        df.write \
          .format("delta") \
          .mode("overwrite") \
          .partitionBy("partition_date") \
          .option("path", bronze_path) \
          .saveAsTable(bronze_table_name)
        log(f"Created and loaded Bronze table {bronze_table_name}")


# Main pipeline
for table in source_tables:
    try:
        log(f"Starting pipeline for {table}")

        # 1. Read incremental/updated records
        df = read_incremental(table)

        if df.limit(1).count() == 0:
            log(f"No new or updated data for {table}")
            continue

        # 2. Load RAW (append-only)
        load_raw(table, df)

        # 3. Upsert into Bronze
        upsert_bronze(table, df)

        log(f"Completed pipeline for {table}")

    except Exception as e:
        log(f"Error processing {table}: {str(e)}")

In [0]:
%sql
select * from ddddddddddd.raw.orders

In [0]:
spark.sql("""
CREATE TABLE timestamp_id
(
    name string,
    timestamp_col TIMESTAMP,
    id_col INT
)
USING DELTA
LOCATION 'abfss://divyacontainer@divyastorrage.dfs.core.windows.net/practice'
""")

In [0]:
spark.sql("""
INSERT INTO ddddddddddd.default.timestamp_id (name, timestamp_col, id_col)
VALUES
  ('Alice', current_timestamp(), 1),
  ('Bob', current_timestamp(), 2),
  ('Charlie', current_timestamp(), 3)
""")

In [0]:
%sql
select * from ddddddddddd.default.timestamp_id

In [0]:
from pyspark.sql.functions import col
df = spark.read.table("ddddddddddd.default.timestamp_id")
df = df.withColumn("timestamp_col", col("timestamp_col").cast("date"))
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("timestamp_id")

In [0]:
df1=spark.sql("describe timestamp_id")
df1.show()

In [0]:
%sql DESCRIBE HISTORY timestamp_id

In [0]:
%sql
SELECT * 
FROM timestamp_id VERSION AS OF 1;


In [0]:
%sql
SELECT * 
FROM timestamp_id VERSION AS OF 2;

In [0]:
%sql
SELECT * 
FROM timestamp_id;